# 灯条 ROI & 方差图 数据查看器

纯 matplotlib 显示，无需 GUI，适合服务器端 Jupyter。

In [13]:
import os, sys
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from ipywidgets import interact, IntSlider, Button, VBox, HBox, Output
from IPython.display import clear_output, display

sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

from light_corner_corrector import LightCornerCorrector
from lable_generator import (TraditionalArmorDetector,
                             extract_number_rois, _compute_expand_factor)

In [14]:
# ====================== 配置路径 ======================
LIGHT_DATA_DIR = "/home/frank/RM2026/python_refactor/test/light_data"
DATASET_DIR    = "/home/frank/RM2026/python_refactor/dataset/competation"

gray_dir = os.path.join(LIGHT_DATA_DIR, 'gray_roi')
var_dir  = os.path.join(LIGHT_DATA_DIR, 'variance_map')
lbl_dir  = os.path.join(LIGHT_DATA_DIR, 'labels')

In [15]:
def load_labels(lbl_dir):
    """解析 labels 目录，返回 (stem, armor_idx, light_idx, tn, bn, axis_p1, axis_p2) 列表。"""
    samples = []
    for f in sorted(os.listdir(lbl_dir)):
        if not f.endswith('.txt'):
            continue
        path = os.path.join(lbl_dir, f)
        with open(path) as fh:
            lines = fh.read().strip().split('\n')
        
        # 第 1 行：角点
        if len(lines) < 1 or not lines[0]:
            continue
        parts = lines[0].split()
        if len(parts) < 4:
            continue
        tn = (float(parts[0]), float(parts[1]))
        bn = (float(parts[2]), float(parts[3]))
        
        # 第 2 行：主轴线段
        axis_p1 = None
        axis_p2 = None
        if len(lines) >= 2 and lines[1]:
            axis_parts = lines[1].split()
            if len(axis_parts) >= 4:
                axis_p1 = (float(axis_parts[0]), float(axis_parts[1]))
                axis_p2 = (float(axis_parts[2]), float(axis_parts[3]))
        
        base = f.replace('.txt', '')
        idx = base.rfind('_armor')
        if idx < 0:
            continue
        stem = base[:idx]
        rest = base[idx + 1:]
        try:
            armor_str, light_str = rest.split('_')
            armor_idx = int(armor_str.replace('armor', ''))
            light_idx = int(light_str.replace('light', ''))
        except ValueError:
            continue
        samples.append((path, stem, armor_idx, light_idx, tn, bn, axis_p1, axis_p2))
    return samples


def find_image(dataset_dir, stem):
    for root, dirs, files in os.walk(dataset_dir):
        for f in files:
            if f == f"{stem}.bmp":
                return os.path.join(root, f)
    return None

In [16]:
# ====================== 加载样本 & 初始化检测器 ======================
samples = load_labels(lbl_dir)
print(f"Loaded {len(samples)} samples")

detector = TraditionalArmorDetector(binary_thresh=80)
detector.l_params.min_ratio = 0.005
detector.l_params.max_ratio = 0.8
detector.l_params.max_angle = 60
detector.l_params.min_length = 6
detector.l_params.min_width = 1
detector.a_params.min_light_ratio = 0.5
detector.a_params.min_small_center_distance = 0.5
detector.a_params.max_small_center_distance = 5.0
detector.a_params.min_large_center_distance = 2.0
detector.a_params.max_large_center_distance = 8.0
detector.a_params.max_angle = 60

corrector = LightCornerCorrector()
cache = {}

Loaded 46 samples


In [17]:
def draw_sample(idx):
    """绘制单样本：原图 + gray ROI + variance map。
    
    标注：红点=角点  绿点=形心  蓝线=主轴线段
    """
    import matplotlib.patches as mpatches

    _, stem, armor_idx, light_idx, tn, bn, axis_p1, axis_p2 = samples[idx]
    
    # 缓存原图检测
    if stem not in cache:
        img_path = find_image(DATASET_DIR, stem)
        if img_path is None:
            print(f"Image not found: {stem}.bmp")
            return
        bayer_raw = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if bayer_raw is None:
            return
        _, _, gray_img, armors, _ = detector.detect(bayer_raw)
        cache[stem] = (bayer_raw, gray_img, armors)
    else:
        bayer_raw, gray_img, armors = cache[stem]
    
    if armor_idx >= len(armors):
        print(f"armor_idx {armor_idx} out of range")
        return
    
    l1, l2 = armors[armor_idx]
    lights = [l1, l2]
    if light_idx >= len(lights):
        print(f"light_idx {light_idx} out of range")
        return
    
    target_light = lights[light_idx]
    other_light = lights[1 - light_idx]
    expand = _compute_expand_factor(target_light.average_brightness,
                                     other_light.average_brightness)
    result = corrector.correct_corners(target_light, gray_img, bayer_raw, expand)
    variance_roi, _, _, axis, top_c, bot_c = result
    if variance_roi is None:
        print(f"corrector early exit (len={target_light.length:.0f})")
        return
    
    bx, by, bw, bh = corrector.extractor.expanded_bbox
    gray_roi = gray_img[by:by + bh, bx:bx + bw]
    roi_h, roi_w = gray_roi.shape
    
    # --- 创建显示图 ---
    fig, axes = plt.subplots(1, 3, figsize=(18, 8))
    fig.suptitle(f"{stem}  armor={armor_idx}  light={light_idx}  [{idx+1}/{len(samples)}]",
                 fontsize=13, fontweight='bold')
    
    # ── 左：原图 ──
    full_rgb = cv2.cvtColor(bayer_raw, cv2.COLOR_BayerBG2RGB)
    axes[0].imshow(full_rgb)
    # bbox
    rect = mpatches.Rectangle((bx, by), bw, bh, fill=False, edgecolor='lime', linewidth=1.5)
    axes[0].add_patch(rect)
    # 角点 & 形心 & 主轴线段
    tl = axis.top_left
    if top_c is not None:
        axes[0].plot(top_c[0], top_c[1], 'o', color='red', markersize=6)
    if bot_c is not None:
        axes[0].plot(bot_c[0], bot_c[1], 'o', color='red', markersize=6)
    center = (axis.centroid[0] + tl[0], axis.centroid[1] + tl[1])
    axes[0].plot(center[0], center[1], 'o', color='lime', markersize=5)
    # 主轴线段
    half_len = target_light.length / 2.0 * 1.2
    dx, dy = axis.direction
    cx_local, cy_local = axis.centroid
    p1 = (cx_local - half_len * dx + tl[0], cy_local - half_len * dy + tl[1])
    p2 = (cx_local + half_len * dx + tl[0], cy_local + half_len * dy + tl[1])
    axes[0].plot([p1[0], p2[0]], [p1[1], p2[1]], color='deepskyblue', linewidth=2)
    
    axes[0].set_title("Full image\n— bbox  — axis  ● corners")
    axes[0].axis('off')
    
    # ── 辅助：在 ROI 面板上标注（matplotlib 原生，不污染像素）──
    def annotate_roi(ax, img, title):
        if len(img.shape) == 2:
            ax.imshow(img, cmap='gray')
        else:
            ax.imshow(img)
        h, w = img.shape[:2]
        # 角点
        if len(tn) == 2:
            ax.plot(tn[0] * w, tn[1] * h, 'o', color='red', markersize=8)
            ax.plot(bn[0] * w, bn[1] * h, 'o', color='red', markersize=8)
        # 主轴线段
        if axis_p1 is not None and axis_p2 is not None:
            ax.plot([axis_p1[0] * w, axis_p2[0] * w],
                    [axis_p1[1] * h, axis_p2[1] * h],
                    color='deepskyblue', linewidth=2)
        ax.set_title(title + "\n— axis  ● corners")
        ax.axis('off')
    
    annotate_roi(axes[1], gray_roi, "gray ROI")
    annotate_roi(axes[2], variance_roi.astype(np.uint8), "variance map")
    
    plt.tight_layout()
    plt.show()

In [ ]:
# ====================== 交互式导航 ======================
out = Output()

slider = IntSlider(min=0, max=len(samples) - 1, step=1, value=0,
                   description='Index', continuous_update=False,
                   layout={'width': '500px'})

btn_prev = Button(description='◀ Prev', button_style='info',
                  layout={'width': '90px'})
btn_next = Button(description='Next ▶', button_style='info',
                  layout={'width': '90px'})
label = Button(description=f'0 / {len(samples) - 1}', disabled=True,
               layout={'width': '120px'})

def on_slider_change(change):
    with out:
        clear_output(wait=True)
        idx = change['new']
        draw_sample(idx)
        label.description = f'{idx} / {len(samples) - 1}'

def on_prev_click(b):
    slider.value = max(0, slider.value - 1)

def on_next_click(b):
    slider.value = min(len(samples) - 1, slider.value + 1)

slider.observe(on_slider_change, names='value')
btn_prev.on_click(on_prev_click)
btn_next.on_click(on_next_click)

display(VBox([HBox([btn_prev, btn_next, label]), slider, out]))

# 显示第一张
with out:
    draw_sample(0)

## 标注图例

| 颜色 | 含义 |
|------|------|
| 蓝色线 | PCA 主轴线段（沿灯条对称方向，覆盖全长 +20%） |
| 红色点 | top / bottom 角点 |
| 绿色框 | expanded_bbox（灯条 ROI 在原图中的范围） |
| 绿色点 | 形心（加权 PCA 中心） |